<a href="https://colab.research.google.com/github/YardenNahum/FinalProjectXAI/blob/main/Phase-B-26-1-R3/Code/AI%20Models/LLMModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports and Initial Installtion Of Libraries**

In [ ]:
!pip install lime
!pip install shap
!pip install dice-ml
!pip install firebase requests beautifulsoup4 nltk --quiet

from firebase import firebase
import json

import kagglehub
import pandas as pd
import os
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import MaxAbsScaler
import numpy as np

import lime
from lime.lime_text import LimeTextExplainer
import matplotlib.pyplot as plt

import shap

import dice_ml
import pandas as pd

import re
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=d709bcf371aae2f616c5bb85e20b3ee52c068c2a87c74a4306a1a25d674fccf1
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.1 MB/s eta 0:00:00


**Database Connection**

In [ ]:
DB_URL = 'https://xai-project-b65cc-default-rtdb.firebaseio.com/'
FBconn = firebase.FirebaseApplication(DB_URL, None)

**DOWNLOAD AND LOAD the Datast from Kaggle**

In [ ]:
#Download the LLM Dataset
path_llm = kagglehub.dataset_download("sunilthite/llm-detect-ai-generated-text-dataset")
#Load the Dataset to path
file_name = "Training_Essay_Data.csv"
full_path = os.path.join(path_llm, file_name)

#Load into DataFrame
df_text = pd.read_csv(full_path)

print("Part 1 Complete: Text Dataset Loaded!")
print(f"Total Essays: {len(df_text)}")
print(f"AI-Generated count: {df_text['generated'].sum()}") # 1 = AI, 0 = Human

100%|██████████| 18.6M/18.6M [00:00<00:00, 78.8MB/s]

Extracting files...


Part 1 Complete: Text Dataset Loaded!
Total Essays: 29145
AI-Generated count: 11637


**Data Preprocessing**

In [ ]:
import re
import numpy as np
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin

FEATURE_DISPLAY_NAMES = {
    "avg_sentence_length": "Average Sentence Length",       # Average number of words per sentence
    "std_sentence_length": "Sentence Length Variation",     # How much sentence lengths vary (uniform vs diverse)
    "type_token_ratio": "Vocabulary Diversity",             # Ratio of unique words - measures richness of vocabulary
    "transition_ratio": "Use of Formal Connectors",         # Frequency of words like "however", "therefore", "for example"
    "comma_ratio": "Comma Usage",                           # Commas per word → structured writing indicator
    "exclamation_ratio": "Exclamation Usage",               # Emotional / expressive punctuation
    "stopword_ratio": "Use of Common Words",                # Frequency of common words like "the", "is"
    "avg_word_length": "Average Word Length",               # Indicates formality / complexity of vocabulary
    "pronoun_ratio": "First-Person Pronouns"                # Use of "I", "me", "we" (Humans use more)
}

# Custom transformer that extracts stylistic (non-lexical) features from text
class StyleFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Internal feature names
        self.feature_names_ = [
            "avg_sentence_length",
            "std_sentence_length",
            "type_token_ratio",
            "transition_ratio",
            "comma_ratio",
            "exclamation_ratio",
            "stopword_ratio",
            "avg_word_length",
            "pronoun_ratio"
        ]

        # Common function words
        self.stopwords_ = {
            "the","a","an","and","or","but","if","while","is","are","was","were",
            "be","been","being","to","of","in","on","for","with","as","by","at",
            "from","that","this","it","its","into","about","than","then","so"
        }

        # Words often overused in AI-generated text (connectors)
        self.transition_words_ = {
            "however", "therefore", "moreover", "furthermore",
            "thus", "overall", "consequently", "additionally",
            "finally", "for example", "in conclusion"
        }

        # First-person pronouns (typically avoided by standard LLMs)
        self.pronouns_ = {
            "i", "me", "my", "mine", "we", "us", "our", "ours"
        }

    def fit(self, X, y=None):
        return self

    # Safe division to avoid division by zero errors
    def _safe_div(self, a, b):
        return a / b if b else 0.0

    # Extract features from a single text instance
    def _extract_one(self, text):
        if not isinstance(text, str):
            text = "" if text is None else str(text)

        text_lower = text.lower()

        # words and sentences
        words = re.findall(r"\b\w+\b", text_lower)
        sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

        num_words = len(words)

        # Sentences length
        sentence_lengths = [len(re.findall(r"\b\w+\b", s)) for s in sentences]

        # Average sentence length
        avg_sentence_length = np.mean(sentence_lengths) if sentence_lengths else 0.0

        # Variation in sentence length
        std_sentence_length = np.std(sentence_lengths) if sentence_lengths else 0.0

        # word diversity
        unique_words = len(set(words))

        # Ratio of unique words
        type_token_ratio = self._safe_div(unique_words, num_words)

        # Transition usage - counting directly from the string to catch multi-word phrases
        transition_count = sum(text_lower.count(t) for t in self.transition_words_)

        # Frequency of connectors
        transition_ratio = self._safe_div(transition_count, num_words)

        # Punctuation features
        comma_ratio = self._safe_div(text.count(","), num_words)        # structured writing
        exclamation_ratio = self._safe_div(text.count("!"), num_words)  # emotional tone

        # Function word usage
        stopword_count = sum(1 for w in words if w in self.stopwords_)
        stopword_ratio = self._safe_div(stopword_count, num_words)

        # First-person pronoun usage
        pronoun_count = sum(1 for w in words if w in self.pronouns_)
        pronoun_ratio = self._safe_div(pronoun_count, num_words)

        # Word length
        avg_word_length = np.mean([len(w) for w in words]) if words else 0.0

        # Return features (must match order of self.feature_names_)
        return [
            avg_sentence_length,
            std_sentence_length,
            type_token_ratio,
            transition_ratio,
            comma_ratio,
            exclamation_ratio,
            stopword_ratio,
            avg_word_length,
            pronoun_ratio
        ]

    def transform(self, X):
        # Apply feature extraction to all texts
        features = np.array([self._extract_one(text) for text in X], dtype=float)

        # Convert to sparse matrix for sklearn
        return sp.csr_matrix(features)

    def get_feature_names_out(self, input_features=None):
        # Return feature names
        return np.array(self.feature_names_, dtype=object)

In [ ]:
# Prediction target
target = "generated"

# text data
X = df_text["text"].astype(str)

# Target
y = df_text[target]

print("Number of samples:", len(X))
print("Target:", target)

Number of samples: 29145
Target: generated


**Model Training with Stratified K-Fold**

In [ ]:
# Cross-validation
#Split the data into 5 parts while keeping the same class distribution
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#Store accuracy scores from each fold
scores = []
feature_pipeline = FeatureUnion([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english",
            max_features=500,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "style",
        Pipeline([
            ("style_extractor", StyleFeatureExtractor()),
            ("scaler", MaxAbsScaler())
        ])
    )
])


print("Training the model")

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    #Build a pipeline that includes text processing and classification
    fold_model = Pipeline([
    ("features", feature_pipeline),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=0.5
    ))
    ])

    #Split the data into training and testing sets
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    #Training data
    fold_model.fit(X_train, y_train)
    #Testing Data
    score = fold_model.score(X_test, y_test)

    #Save accuracy score
    scores.append(score)
    print(f"Fold {fold}: {score:.2%}")

print(f"\nAverage Project Accuracy: {np.mean(scores):.4f}")

# Final model
model = Pipeline([
    ("features", feature_pipeline),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=0.5
    ))
])

model.fit(X, y)

Training the model
Fold 1: 97.75%
Fold 2: 98.35%
Fold 3: 97.94%
Fold 4: 98.37%
Fold 5: 98.25%

Average Project Accuracy: 0.9813


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('tfidf',
                                                 TfidfVectorizer(max_features=500,
                                                                 min_df=2,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 stop_words='english')),
                                                ('style',
                                                 Pipeline(steps=[('style_extractor',
                                                                  StyleFeatureExtractor()),
                                                                 ('scaler',
                                                                  MaxAbsScaler())]))])),
                ('classifier',
                 LogisticRegression(C=0.5, class_weight='balanced',
                                    max_iter=1000))])

**Creating SHAP, LIME, and DiCE**

In [ ]:
# Create folder if it does not exist
def Ensure_Directory_Exists(folder_path):
    os.makedirs(folder_path, exist_ok=True)

**Generate LIME**

In [ ]:
def Generate_Lime_Explanation(text_instance, model, record_id, output_dir="plots/llm"):
    # Create output folder if it does not exist
    Ensure_Directory_Exists(output_dir)

    # LIME explains the full model by perturbing the input text
    # and checking how the model's prediction changes
    explainer = LimeTextExplainer(
        class_names=["Human", "AI"]
    )

    # Generate explanation for this specific text instance
    exp = explainer.explain_instance(
        text_instance,
        model.predict_proba,   # uses the real hybrid pipeline
        num_features=20
    )

    # Extract important words/phrases and their influence weights
    lime_features = [
        {
            "feature": feature,
            "weight": round(float(weight), 4)
        }
        for feature, weight in exp.as_list()
    ]

    # Save LIME explanation as an interactive HTML file
    lime_html_path = os.path.join(output_dir, f"lime_record_{record_id}.html")
    exp.save_to_file(lime_html_path)
    local_probs = exp.predict_proba

    return {
        "Original_Text": text_instance,
        "Prediction":{
          "Human": round(float(local_probs[0]), 4),
          "AI": round(float(local_probs[1]), 4)},
        "features": lime_features,
        "Type": "Textual",
        "html_path": lime_html_path
    }

**Generate SHAP**

In [ ]:
def Generate_Shap_Explanation(text, X, model, record_id, output_dir="plots/llm"):
    Ensure_Directory_Exists(output_dir)

    # Get model parts
    feature_transformer = model.named_steps["features"]
    classifier = model.named_steps["classifier"]

    # Transform full dataset using the real model features
    X_transformed = feature_transformer.transform(X)

    # Transform input text
    text_transformed = feature_transformer.transform([text])

    # Prediction
    prediction = int(classifier.predict(text_transformed)[0])
    prediction_proba = float(classifier.predict_proba(text_transformed)[0][prediction])

    # SHAP explainer on the classifier
    explainer = shap.LinearExplainer(classifier, X_transformed)

    shap_values = explainer.shap_values(text_transformed)

    if isinstance(shap_values, list):
        weights = shap_values[prediction][0]
        base_val = float(explainer.expected_value[prediction])
    else:
        weights = shap_values[0]
        base_val = float(explainer.expected_value)

    # Get feature names
    tfidf_names = feature_transformer.transformer_list[0][1].get_feature_names_out()
    style_pipe = feature_transformer.transformer_list[1][1]
    style_names = style_pipe.named_steps["style_extractor"].get_feature_names_out()

    feature_names = np.concatenate([tfidf_names, style_names])

    # get names for json
    def clean_feature_name(name):
        return FEATURE_DISPLAY_NAMES.get(name, name)

    # Build explanation object
    explanation = shap.Explanation(
        values=weights,
        base_values=base_val,
        data=text_transformed.toarray()[0],
        feature_names=[clean_feature_name(f) for f in feature_names]
    )

    # Save SHAP plot
    shap_png_path = os.path.join(output_dir, f"shap_record_{record_id}.png")

    plt.figure()
    shap.plots.bar(explanation, max_display=10, show=False)
    plt.savefig(shap_png_path, bbox_inches="tight")
    plt.close()

    # Build JSON result
    shap_json = {
        "base_value": round(base_val, 4),
        "prediction": "AI" if prediction == 1 else "Human",
        "prediction_probability": round(prediction_proba, 4),
        "feature_impacts": [
            {
                "feature": clean_feature_name(feature),
                "weight": round(float(weight), 4)
            }
            for feature, weight in zip(feature_names, weights)
            if abs(weight) > 0.01
        ],
        "png_path": shap_png_path
    }

    return shap_json

**Generate DiCE**

In [ ]:
def create_dice_json(dice_response, desired_class, style_feature_names, classifier, current_target_prob):
    cf_example = dice_response.cf_examples_list[0]
    cf_df = cf_example.final_cfs_df.copy()
    original_df = cf_example.test_instance_df.copy()
    feature_columns = [c for c in original_df.columns if c != "label"]
    original_row = original_df.iloc[0]

    target_label = "Human" if desired_class == 0 else "AI"

    def feature_explanation(raw_feature, direction):
        explanations = {
            "avg_sentence_length": {"increase": "Use slightly longer sentences.", "decrease": "Use shorter sentences."},
            "std_sentence_length": {"increase": "Vary sentence lengths more.", "decrease": "Make sentence lengths more uniform."},
            "type_token_ratio": {"increase": "Use a wider variety of words.", "decrease": "Reuse words more often."},
            "transition_ratio": {"increase": "Use more formal connectors (e.g. 'however').", "decrease": "Use fewer formal connectors."},
            "comma_ratio": {"increase": "Use more commas.", "decrease": "Use fewer commas."},
            "exclamation_ratio": {"increase": "Use more expressive punctuation (!).", "decrease": "Use less expressive punctuation."},
            "space_ratio": {"increase": "Use more spacing.", "decrease": "Make spacing tighter."},
            "stopword_ratio": {"increase": "Use more common connecting words.", "decrease": "Use fewer common connecting words."},
            "avg_word_length": {"increase": "Use slightly longer words.", "decrease": "Use shorter words."},
            "pronoun_ratio": {"increase": "Use more personal language (e.g. 'I', 'we').", "decrease": "Use less personal language."}
        }
        return explanations.get(raw_feature, {}).get(direction)

    # Initialize JSON with the current probability
    result_json = {
        "current_target_probability": f"{current_target_prob * 100:.1f}%"
    }

    for i in range(len(cf_df)):
        cf_row = cf_df.iloc[i]

        # gain and probability
        cf_features_only = cf_row[feature_columns].to_frame().T
        cf_probs = classifier.predict_proba(cf_features_only)[0]
        cf_target_prob = cf_probs[desired_class]
        potential_gain = cf_target_prob - current_target_prob
        changes = []
        for feature in feature_columns:
            if feature not in style_feature_names:
                continue

            try:
                old_val = float(original_row[feature])
                new_val = float(cf_row[feature])
            except:
                continue

            diff = new_val - old_val
            if abs(diff) < 1e-4:
                continue

            direction = "increase" if diff > 0 else "decrease"
            suggestion = feature_explanation(feature, direction)

            if suggestion:
                changes.append({
                    "feature": feature,
                    "original_value": round(old_val, 3),
                    "new_value": f"{suggestion} ({round(new_val, 3)})", # Suggestion + Value
                })

        if changes:
            result_json[f"option_{i+1}"] = {
                "new_target": target_label,
                "target_probability": f"{cf_target_prob * 100:.1f}%",
                "potential_gain": f"+{potential_gain * 100:.1f}%",
                "Changes": changes
            }

    return result_json

In [ ]:
def Generate_Dice_Explanation(query_text, X, y, model, total_CFs=1):
    feature_transformer = model.named_steps["features"]
    classifier = model.named_steps["classifier"]

    tfidf_names = feature_transformer.transformer_list[0][1].get_feature_names_out()
    style_pipe = feature_transformer.transformer_list[1][1]
    style_names = style_pipe.named_steps["style_extractor"].get_feature_names_out()
    feature_names = list(np.concatenate([tfidf_names, style_names]))

    full_X_transformed = feature_transformer.transform(X)

    df = pd.DataFrame(full_X_transformed.toarray(), columns=feature_names, index=X.index)
    df["label"] = y.values

    dice_data = dice_ml.Data(dataframe=df, continuous_features=feature_names, outcome_name="label")
    dice_model = dice_ml.Model(model=classifier, backend="sklearn")
    explainer = dice_ml.Dice(dice_data, dice_model, method="genetic")

    query_vector = feature_transformer.transform([query_text]).toarray()
    query_df = pd.DataFrame(query_vector, columns=feature_names)

    current_prediction = int(classifier.predict(query_df)[0])
    probabilities = classifier.predict_proba(query_df)[0]

    desired_class = 1 - current_prediction
    current_target_prob = probabilities[desired_class]

    dice_response = explainer.generate_counterfactuals(
        query_df,
        total_CFs=total_CFs,
        desired_class=desired_class,
        features_to_vary=list(style_names)
    )

    return create_dice_json(
        dice_response=dice_response,
        desired_class=desired_class,
        style_feature_names=set(style_names),
        classifier=classifier,
        current_target_prob=current_target_prob
    )

**Save prediction and explnations to database**

In [ ]:
def Save_Report(system_name, report):
    try:
        record_id = report["Prediction"]["record_id"]
        FBconn.put(f"/Reports/{system_name}", str(record_id), report)
        print(f"Report saved successfully in {system_name} for record {record_id}.")
    except Exception as e:
        print(f"Firebase update error: {e}")

In [ ]:
def Generate_Text_Prediction_Report(test_idx, X_text, y, full_model):
    warnings.filterwarnings("ignore")

    # Select one text sample from the dataset
    text_sample = X_text.iloc[test_idx]

    # Predict class for this text
    # 0 = Human, 1 = AI
    prediction = int(full_model.predict([text_sample])[0])
    prediction_proba = full_model.predict_proba([text_sample])[0]


    # Store original input text
    raw_data = [
        {
            "feature": "Text",
            "value": str(text_sample)
        }
    ]
    # Generate LIME explanation
    lime_result = Generate_Lime_Explanation(
        text_sample,
        full_model,
        record_id=test_idx
    )

    # Generate SHAP explanation
    shap_result = Generate_Shap_Explanation(
        text_sample,
        X_text,
        full_model,
        record_id=test_idx
    )

    # Generate DiCE counterfactual explanation
    dice_result = Generate_Dice_Explanation(
        text_sample,
        X_text,
        y,
        full_model
    )


    # Combine all outputs into one final report
    report = {
        "Prediction": {
            "record_id": int(test_idx),
            "prediction": "AI" if prediction == 1 else "Human",
            "description": "This AI system analyzes text patterns to predict if the content was generated by a AI or written by a human."
        },
        "Original_Data": raw_data,
        "explanations": {
            "lime": lime_result,
            "shap": shap_result,
            "dice": dice_result
        }
    }

    return report

In [ ]:
report = Generate_Text_Prediction_Report(1000, X, y, model)
Save_Report("LLM_Report", report)

print(json.dumps(report, indent=2))
print(report["explanations"]["lime"]["html_path"])
print(report["explanations"]["shap"]["png_path"])

100%|██████████| 1/1 [00:03<00:00,  3.86s/it]


Report saved successfully in LLM_Report for record 1000.
{
  "Prediction": {
    "record_id": 1000,
    "prediction": "Human",
    "description": "This AI system analyzes text patterns to predict if the content was generated by a AI or written by a human."
  },
  "Original_Data": [
    {
      "feature": "Text",
      "value": "Are cars beginning to go out of fashion? Over the last few years, fewer and fewer people are getting their driver's license, and less people are buying cars. That's because walking or taking public transportation where you need to go can be more beneficial than driving. Using alternative transportation decreases stress, lowers air pollution, and eliminates the cost of owning a car. These reasons are causing people to eliminate personal vehicles out of their lives.\n\nTo begin, using alternative transportation can lower stress. Millions of people around the world face the same problem every morning and afternoon: traffic. Traffic jams cause people to be late for 